# [12-5강] MLP baseline 준비 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. 이미지 batch flatten하기

MLP는 2D feature vector를 입력으로 받기 때문에 이미지 batch를 `[N, C*H*W]`로 펼칩니다.

In [2]:
x = torch.randn(10, 1, 8, 8)
# TODO: batch 차원을 유지한 채 flatten하세요.
flat = x.view(x.size(0), -1)
print(flat.shape)


torch.Size([10, 64])


## 문제 2. MLP baseline 모델 만들기

flatten된 64차원 입력을 받아 2개 class logits를 출력하는 MLP를 만듭니다.

In [3]:
# TODO: 64 -> 16 -> 2 구조로 수정하세요.
mlp = nn.Sequential(
    nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 2    )
)
x = torch.randn(5, 1, 8, 8)
flat = x.view(x.size(0), -1)
try:
    print(mlp(flat).shape)
except RuntimeError as e:
    print('입출력 차원을 다시 확인하세요:', str(e).split('\\n')[0])


torch.Size([5, 2])


## 문제 3. MLP 한 epoch 학습하기

toy 이미지 데이터를 flatten한 뒤 MLP를 한 epoch 학습합니다.

In [10]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

mlp = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.02)

def train_mlp_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # TODO: x를 flatten하고 학습 step을 완성하세요.
        flat = x.view(x.size(0),-1)
        optimizer.zero_grad()
        logits = model(flat)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
    return total_loss / total
print('mlp loss:', train_mlp_one_epoch(mlp, train_loader))


mlp loss: 0.33131508231163026


In [11]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

mlp = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.02)

def train_mlp_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        flat = x.view(x.size(0), -1)
        optimizer.zero_grad()
        logits = model(flat)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
    return total_loss / total
print('mlp loss:', train_mlp_one_epoch(mlp, train_loader))


mlp loss: 0.39479590952396393
